<a href="https://colab.research.google.com/github/FrancisJAlboroto/Final-Project-CS2/blob/main/Dahlia_Library_NB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import requests

url = "https://raw.githubusercontent.com/FrancisJAlboroto/Final-Project-CS2/refs/heads/main/data/library.json"
response = requests.get(url)
data = response.json()

for borrower in data:
    print(f"\nBorrower ID: {borrower['borrower_id']}")
    print(f"Student: {borrower['student']}")
    print("Books:")

    for book in borrower["books"]:
        print(f"  - Title: {book['book_title']}")
        print(f"    Author: {book['author']}")
        print(f"    Borrow Date: {book['borrow_date']}")
        print(f"    Return Date: {book['return_date']}")


Borrower ID: 1
Student: Carlos Mendoza
Books:
  - Title: Introduction to Python
    Author: Guido van Rossum
    Borrow Date: 2025-08-20
    Return Date: 2025-09-01
  - Title: Advanced Algebra
    Author: Richard Brown
    Borrow Date: 2025-09-10
    Return Date: 2025-09-20
  - Title: World History Volume 1
    Author: Arthur Kemp
    Borrow Date: 2025-09-15
    Return Date: None

Borrower ID: 2
Student: Ella Cruz
Books:
  - Title: Data Science for Beginners
    Author: Jane Smith
    Borrow Date: 2025-09-05
    Return Date: 2025-09-12
  - Title: Creative Writing Essentials
    Author: Maria Johnson
    Borrow Date: 2025-09-10
    Return Date: None
  - Title: Physics Fundamentals
    Author: Isaac Newton
    Borrow Date: 2025-09-18
    Return Date: None

Borrower ID: 3
Student: Mark Dela Peña
Books:
  - Title: The Great Gatsby
    Author: F. Scott Fitzgerald
    Borrow Date: 2025-09-07
    Return Date: None
  - Title: Introduction to Philosophy
    Author: Socrates
    Borrow Date: 20

In [ ]:
!pip install firebase-admin


In [8]:
import firebase_admin
from firebase_admin import credentials, db

if not firebase_admin._apps:
    cred = credentials.Certificate("/content/firebase_key.json")
    firebase_admin.initialize_app(cred, {
     "databaseURL": "https://dahlia-library-db-default-rtdb.asia-southeast1.firebasedatabase.app/"
    })
    print("Firebase connected successfully!")
else:
    print("Firebase app already initialized.")

Firebase app already initialized.


In [9]:
import requests
import json
import firebase_admin
from firebase_admin import credentials, db

# Initialize Firebase if not already initialized
if not firebase_admin._apps:
    try:
        # Ensure the firebase_key.json file is uploaded to /content/
        cred = credentials.Certificate("/content/firebase_key.json")
        firebase_admin.initialize_app(cred, {
            "databaseURL": "https://dahlia-library-db-default-rtdb.asia-southeast1.firebasedatabase.app/"
        })
        print("Firebase initialized successfully!")
    except Exception as e:
        print(f"Failed to initialize Firebase: {e}")
else:
    print("Firebase app already initialized.")

url = "https://raw.githubusercontent.com/FrancisJAlboroto/Final-Project-CS2/refs/heads/main/data/library.json"
response = requests.get(url)

if response.ok:
    data = response.json()
    print("JSON file loaded")

    ref = db.reference("borrowers")

    for borrower in data:
        borrower_id = borrower["borrower_id"]
        ref.child(str(borrower_id)).set(borrower)

    print("Data uploaded successfully!")
else:
    print(f"Error loading JSON file: {response.status_code} - {response.reason}")
    print(response.text)

Firebase app already initialized.
JSON file loaded
Data uploaded successfully!


In [10]:
from firebase_admin import db

ref = db.reference("borrowers")
data = ref.get()

print("Data loaded successfully!")

Data loaded successfully!


In [13]:
import firebase_admin
from firebase_admin import credentials, db

if not firebase_admin._apps:
    try:
        cred = credentials.Certificate("/content/firebase_key.json")
        firebase_admin.initialize_app(cred, {
            "databaseURL": "https://dahlia-library-db-default-rtdb.asia-southeast1.firebasedatabase.app/"
        })
        print("Firebase initialized successfully!")
    except Exception as e:
        print(f"Failed to initialize Firebase: {e}")
else:
    print("Firebase app already initialized.")


while True:
    ref = db.reference("borrowers")
    data = ref.get()

    if data is None:
        print("\nError: No data found in the database. Please upload your JSON first.")
        break

    print("\n--- Library System ---")
    print("1 - Show Unreturned Books")
    print("2 - Count Student Borrowed Books")
    print("3 - Total Books Borrowed")
    print("4 - Search Book")
    print("5 - Find Who Borrowed a Book")
    print("0 - Exit")


    choice = input("\nEnter choice: ").strip()


    if choice == "1":
        found = False
        for borrower_data in data.values():
            if borrower_data is not None:
                for book in borrower_data.get("books", []):
                    if book.get("return_date") is None:
                        print(f"{borrower_data.get('student', 'Unknown')} - {book.get('book_title', 'Unknown Title')}")
                        found = True

        if not found:
            print("Message: All books have been returned.")


    elif choice == "2":
        name = input("Enter student name: ").strip()


        if not name:
            print("Error: Student name cannot be blank.")
            continue

        found_student = False
        for borrower_data in data.values():
            if borrower_data is not None:
                if name.lower() in borrower_data.get("student", "").lower():
                    count = sum(1 for book in borrower_data.get("books", []) if book.get("return_date") is None)
                    print(f"{borrower_data.get('student')} has {count} book(s) currently borrowed.")
                    found_student = True

        if not found_student:
            print("Error: Student not found in records.")


    elif choice == "3":
        total = sum(len(borrower_data.get("books", [])) for borrower_data in data.values() if borrower_data is not None)
        print("Total books borrowed overall:", total)


    elif choice == "4":
        search = input("Enter title or author: ").strip()


        if not search:
            print("Error: Search query cannot be blank.")
            continue

        found = False
        for borrower_data in data.values():
            if borrower_data is not None:
                for book in borrower_data.get("books", []):
                    title = book.get("book_title", "").lower()
                    author = book.get("author", "").lower()
                    if search.lower() in title or search.lower() in author:
                        print(f"{book.get('book_title')} by {book.get('author')} - Borrowed by {borrower_data.get('student')}")
                        found = True

        if not found:
            print("Message: No books found matching that search.")


    elif choice == "5":
        search = input("Enter book title: ").strip()


        if not search:
            print("Error: Book title cannot be blank.")
            continue

        found = False
        for borrower_data in data.values():
            if borrower_data is not None:
                for book in borrower_data.get("books", []):
                    if search.lower() in book.get("book_title", "").lower():
                        status = "Returned" if book.get("return_date") is not None else "Still Borrowed"
                        print(f"{borrower_data.get('student')} borrowed {book.get('book_title')} [{status}]")
                        found = True

        if not found:
            print("Error: Book not found in records.")

    elif choice == "0":
        print("Exiting system. Goodbye!")
        break

    else:
        print("Error: Invalid choice. Please enter a number between 0 and 5.")

Firebase app already initialized.

--- Library System ---
1 - Show Unreturned Books
2 - Count Student Borrowed Books
3 - Total Books Borrowed
4 - Search Book
5 - Find Who Borrowed a Book
0 - Exit

Enter choice: 1
Carlos Mendoza - World History Volume 1
Ella Cruz - Creative Writing Essentials
Ella Cruz - Physics Fundamentals
Mark Dela Peña - The Great Gatsby
Mark Dela Peña - Modern JavaScript
Sofia Ramos - English Literature Classics
Sofia Ramos - Database Systems
Rui Lopez - Physics 2
Seven Eleven - Store
Francis - poll

--- Library System ---
1 - Show Unreturned Books
2 - Count Student Borrowed Books
3 - Total Books Borrowed
4 - Search Book
5 - Find Who Borrowed a Book
0 - Exit

Enter choice: 2
Enter student name: francis
Francis has 1 book(s) currently borrowed.

--- Library System ---
1 - Show Unreturned Books
2 - Count Student Borrowed Books
3 - Total Books Borrowed
4 - Search Book
5 - Find Who Borrowed a Book
0 - Exit

Enter choice: 3
Total books borrowed overall: 15

--- Library 

In [ ]:
import firebase_admin
from firebase_admin import credentials, db


if not firebase_admin._apps:
    cred = credentials.Certificate("/content/firebase_key.json")
    firebase_admin.initialize_app(cred, {
        "databaseURL": "https://dahlia-library-db-default-rtdb.asia-southeast1.firebasedatabase.app/"
    })
    print("Firebase initialized successfully!")
else:
    print("Firebase already initialized.")


ref = db.reference("borrowers")


while True:
    print("\n===== LIBRARY DATABASE MENU =====")
    print("1. Display All Borrowers")
    print("2. Add New Borrower")
    print("3. Update Borrower")
    print("4. Delete Borrower")
    print("5. Exit")

    choice = input("Enter choice: ")


    if choice == "1":
        data = ref.get()

        if data:
            print("\nBorrower List:")
            for borrower in data.values():
                if borrower:
                    print(f"\nBorrower ID: {borrower['borrower_id']}")
                    print(f"Student: {borrower['student']}")
                    print("Books:")

                    for book in borrower["books"]:
                        print(f"  - Title: {book['book_title']}")
                        print(f"    Author: {book['author']}")
                        print(f"    Borrow Date: {book['borrow_date']}")
                        print(f"    Return Date: {book.get('return_date')}")
                    print("-----------------------------")
        else:
            print("No records found.")


    elif choice == "2":
        bid = input("Enter Borrower ID: ")

        # Check if ID already exists
        if ref.child(bid).get():
            print("Borrower ID already exists!")
            continue

        name = input("Enter Student Name: ")
        books = []

        add_more = "y"
        while add_more.lower() == "y":
            title = input("Enter Book Title: ")
            author = input("Enter Author: ")
            borrow_date = input("Enter Borrow Date (YYYY-MM-DD): ")
            return_date = input("Enter Return Date (leave blank if not returned): ")

            if return_date == "":
                return_date = None

            book = {
                "book_title": title,
                "author": author,
                "borrow_date": borrow_date,
                "return_date": return_date
            }

            books.append(book)
            add_more = input("Add another book? (y/n): ")

        borrower = {
            "borrower_id": int(bid),
            "student": name,
            "books": books
        }

        ref.child(bid).set(borrower)
        print("Borrower added successfully!")


    elif choice == "3":
        bid = input("Enter Borrower ID to update: ")
        borrower_ref = ref.child(bid)
        borrower = borrower_ref.get()

        if borrower:
            print(f"Current Name: {borrower['student']}")
            name = input("Enter new name: ")

            books = []
            add_more = "y"

            while add_more.lower() == "y":
                title = input("Enter Book Title: ")
                author = input("Enter Author: ")
                borrow_date = input("Enter Borrow Date (YYYY-MM-DD): ")
                return_date = input("Enter Return Date (leave blank if not returned): ")

                if return_date == "":
                    return_date = None

                book = {
                    "book_title": title,
                    "author": author,
                    "borrow_date": borrow_date,
                    "return_date": return_date
                }

                books.append(book)
                add_more = input("Add another book? (y/n): ")

            borrower_ref.update({
                "student": name,
                "books": books
            })

            print("Borrower updated successfully!")
        else:
            print("Borrower not found.")


    elif choice == "4":
        bid = input("Enter Borrower ID to delete: ")

        if ref.child(bid).get():
            ref.child(bid).delete()
            print("Borrower deleted successfully!")
        else:
            print("Borrower not found.")


    elif choice == "5":
        print("Exiting program...")
        break

    else:
        print("Invalid choice. Try again.")

Firebase already initialized.

===== LIBRARY DATABASE MENU =====
1. Display All Borrowers
2. Add New Borrower
3. Update Borrower
4. Delete Borrower
5. Exit
Enter choice: 1

Borrower List:

Borrower ID: 1
Student: Carlos Mendoza
Books:
  - Title: Introduction to Python
    Author: Guido van Rossum
    Borrow Date: 2025-08-20
    Return Date: 2025-09-01
  - Title: Advanced Algebra
    Author: Richard Brown
    Borrow Date: 2025-09-10
    Return Date: 2025-09-20
  - Title: World History Volume 1
    Author: Arthur Kemp
    Borrow Date: 2025-09-15
    Return Date: None
-----------------------------

Borrower ID: 2
Student: Ella Cruz
Books:
  - Title: Data Science for Beginners
    Author: Jane Smith
    Borrow Date: 2025-09-05
    Return Date: 2025-09-12
  - Title: Creative Writing Essentials
    Author: Maria Johnson
    Borrow Date: 2025-09-10
    Return Date: None
  - Title: Physics Fundamentals
    Author: Isaac Newton
    Borrow Date: 2025-09-18
    Return Date: None
-----------------

KeyboardInterrupt: Interrupted by user